<a href="https://colab.research.google.com/github/SksuriyaPrakash/1DL/blob/main/llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install  langchain_community langchain-community langchain-text-splitters faiss-cpu sentence-transformers pypdf duckduckgo-search transformers

In [3]:
from google.colab import files
uploaded = files.upload()

Saving test.pdf to test (4).pdf


In [4]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("test.pdf")
documents = loader.load()

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
docs = splitter.split_documents(documents)

In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()

/tmp/ipykernel_72903/4162380786.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/tmp/ipykernel_72903/4162380786.py:3: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# @title Default title text
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(docs,embeddings)


In [8]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=100
)
llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM

In [9]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

In [11]:
def smart_agent(query):
    results = db.similarity_search_with_score(query, k=3)

    good_docs = [doc for doc, score in results if score < 0.8]

    # ✅ If PDF has relevant data
    if good_docs:
        context = " ".join([d.page_content for d in good_docs])

        prompt = f"""
        Context:
        {context}

        Question:
        {query}

        Answer only from context in 2 lines.
        """

        return llm.invoke(prompt).strip()

    # ❌ Else → Web search
    else:
        web_data = search.run(query)

        prompt = f"""
        Use this web data:
        {web_data}

        Question:
        {query}

        Give clear answer in 2 lines.
        """

        return llm.invoke(prompt).strip()

In [12]:
print(smart_agent("today important news"))

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Use this web data:
        NPRnews, audio, and podcasts. Coverage of breaking stories, national and worldnews, politics, business, science, technology, and extended coverage of major national and world events. Reuters.com is your online source for the latest USnewsstories and current events, ensuring our readers up to date with any breakingnewsdevelopments U.S. breakingnews:Today'stop stories updated by the CBSNewsteam. Stay on top ofNewslatest developments on the ground with Al Jazeera's fact-basednews, exclusive video footage, photos and updated maps. Reporting breakingnewsand developing stories in real time; covering the mostimportantstories of the day and taking deep dives on issues.

        Question:
        today important news

        Give clear answer in 2 lines.
